In [12]:
import io
from pathlib import Path

import chess
import chess.pgn
import chess.svg
import chess.engine
import pandas as pd
from PIL import Image
import cairosvg

import ipywidgets as widgets
from IPython.display import display, clear_output

from utils.opening_repository import OpeningRepository
from utils.game_enrichment_transformer import (
    GameEnrichmentTransformer,
    enriched_moves_to_flat_dicts,
)

In [13]:
STOCKFISH_PATH = "/opt/homebrew/bin/stockfish"  # Apple Silicon Mac
# STOCKFISH_PATH = "/usr/local/bin/stockfish"   # Intel Mac

OPENINGS_PATH = "openings_dataset/all.tsv"

ENGINE_DEPTH = 10

In [14]:
from utils.chesscom_repository import ChessComRepository

USERNAME = "bassisw"
MAX_GAMES = 20
SINCE_YEAR = 2025
SINCE_MONTH = 1

In [15]:
chesscom_repo = ChessComRepository(USERNAME)

games = []

for game_data in chesscom_repo.iter_all_games(
    since_year=SINCE_YEAR,
    since_month=SINCE_MONTH,
):
    if "pgn" not in game_data:
        continue

    games.append(game_data)

    if len(games) >= MAX_GAMES:
        break

print(f"Pulled {len(games)} games")

Pulled 20 games


In [16]:
if not games:
    raise RuntimeError("No games found. Check USERNAME, SINCE_YEAR, SINCE_MONTH.")

GAME_INDEX = 0

game_data = games[GAME_INDEX]

print(game_data["url"])
print(game_data.get("time_class"))
print(game_data.get("white"))
print(game_data.get("black"))

https://www.chess.com/game/live/130193958939
blitz
{'rating': 561, 'result': 'timeout', '@id': 'https://api.chess.com/pub/player/bassisw', 'username': 'bassisw', 'uuid': 'fb096f38-31f0-11ee-bdae-cfd83ad0aee0'}
{'rating': 732, 'result': 'win', '@id': 'https://api.chess.com/pub/player/sylvathur', 'username': 'Sylvathur', 'uuid': 'edd9d908-5d48-11eb-95c2-cd8c6efeb2ce'}


In [17]:
opening_repo = OpeningRepository(OPENINGS_PATH)

transformer = GameEnrichmentTransformer(
    stockfish_path=STOCKFISH_PATH,
    opening_repository=opening_repo,
    engine_limit=chess.engine.Limit(depth=ENGINE_DEPTH),
)

enriched_games = transformer.transform_games([game_data])
enriched_game = enriched_games[0]

moves_df = pd.DataFrame(enriched_moves_to_flat_dicts(enriched_games))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

game_info = {
    key: value
    for key, value in vars(enriched_game).items()
    if key != "moves"
}
game_info_df = pd.DataFrame([game_info])

display(game_info_df)

moves_df.head()

,uuid,url,white_username,black_username,white_rating,black_rating,white_result,black_result,result,time_class,time_control,rated,end_time,chesscom_white_accuracy,chesscom_black_accuracy,opening_eco,opening_name
0,40fc279d-cec9-11ef-ad47-6cfe544c0428,https://www.chess.com/game/live/130193958939,bassisw,Sylvathur,561,732,timeout,win,0-1,blitz,180,False,1736455305,None,None,A17,"English Opening: Anglo-Indian Defense, Hedgehog System"


,game_uuid,game_url,ply,move_number,player_color,player_username,san,uci,fen_before,fen_after,eval_before_cp,eval_after_cp,eval_best_move_cp,eval_played_move_cp,move_cp_loss,best_move_uci,top_engine_moves,best_move_cp_gain,win_prob_before,win_prob_after,win_prob_loss,clock_before,clock_after,clock_spent,phase,position_type_tags,tactical_position,quiet_middlegame,complexity,number_of_legal_moves,eval_volatility_among_top_engine_lines,forcing_line_depth,low_gap_between_top_moves,engine_top_move_is_forcing,move_is_threat,is_capture,is_check,is_castling,is_promotion,in_opening_book,opening_eco,opening_name
0,40fc279d-cec9-11ef-ad47-6cfe544c0428,https://www.chess.com/game/live/130193958939,1,1,white,bassisw,c4,c2c4,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1,rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR b KQkq - 0 1,45,12,45,11,34,e2e4,"[{'rank': 1, 'move_uci': 'e2e4', 'eval_cp': 45, 'line_uci': ['e2e4', 'c7c5', 'g1f3', 'e7e6', 'b1c3', 'b8c6', 'd2d4', 'c5d4', 'f3d4', 'g8f6', 'd4c6', 'b7c6']...",8,0.523170,0.506183,0.016987,180.0,180.0,0.0,opening,"[equal, quiet]",False,False,121.092121,20,9.092121,0,92.0,False,False,False,False,False,False,True,A10,English Opening
1,40fc279d-cec9-11ef-ad47-6cfe544c0428,https://www.chess.com/game/live/130193958939,2,1,black,Sylvathur,Nf6,g8f6,rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR b KQkq - 0 1,rnbqkb1r/pppppppp/5n2/8/2P5/8/PP1PPPPP/RNBQKBNR w KQkq - 1 2,15,38,15,41,26,e7e5,"[{'rank': 1, 'move_uci': 'e7e5', 'eval_cp': 15, 'line_uci': ['e7e5', 'b1c3', 'g8f6', 'g1f3', 'b8c6', 'g2g3', 'd7d5', 'c4d5', 'f6d5', 'd2d3']}, {'rank': 2, '...",20,0.491631,0.478809,0.012822,180.0,180.0,0.0,opening,"[equal, quiet]",False,False,110.801234,20,10.801234,0,80.0,False,False,False,False,False,False,True,A15,English Opening: Anglo-Indian Defense
2,40fc279d-cec9-11ef-ad47-6cfe544c0428,https://www.chess.com/game/live/130193958939,3,2,white,bassisw,Nc3,b1c3,rnbqkb1r/pppppppp/5n2/8/2P5/8/PP1PPPPP/RNBQKBNR w KQkq - 1 2,rnbqkb1r/pppppppp/5n2/8/2P5/2N5/PP1PPPPP/R1BQKBNR b KQkq - 2 2,41,20,41,20,21,d2d4,"[{'rank': 1, 'move_uci': 'd2d4', 'eval_cp': 41, 'line_uci': ['d2d4', 'e7e6', 'b1c3', 'f8b4', 'e2e3', 'b4c3', 'b2c3', 'b7b6', 'f2f3', 'd7d6', 'e3e4', 'b8c6',...",5,0.521113,0.510304,0.010809,180.0,178.8,1.2,opening,"[equal, quiet]",False,False,127.338708,22,10.338708,0,95.0,False,False,False,False,False,False,True,A16,"English Opening: Anglo-Indian Defense, Queen's Knight Variation"
3,40fc279d-cec9-11ef-ad47-6cfe544c0428,https://www.chess.com/game/live/130193958939,4,2,black,Sylvathur,e6,e7e6,rnbqkb1r/pppppppp/5n2/8/2P5/2N5/PP1PPPPP/R1BQKBNR b KQkq - 2 2,rnbqkb1r/pppp1ppp/4pn2/8/2P5/2N5/PP1PPPPP/R1BQKBNR w KQkq - 0 3,20,41,20,37,17,e7e5,"[{'rank': 1, 'move_uci': 'e7e5', 'eval_cp': 20, 'line_uci': ['e7e5', 'g1f3', 'b8c6', 'g2g3', 'd7d5', 'c4d5', 'f6d5', 'f1g2', 'd5b6', 'd2d3', 'f8e7', 'c1e3',...",14,0.488842,0.477138,0.011704,180.0,178.8,1.2,opening,"[equal, quiet]",False,False,114.847546,22,6.847546,0,86.0,False,False,False,False,False,False,True,A17,"English Opening: Anglo-Indian Defense, Hedgehog System"
4,40fc279d-cec9-11ef-ad47-6cfe544c0428,https://www.chess.com/game/live/130193958939,5,3,white,bassisw,g3,g2g3,rnbqkb1r/pppp1ppp/4pn2/8/2P5/2N5/PP1PPPPP/R1BQKBNR w KQkq - 0 3,rnbqkb1r/pppp1ppp/4pn2/8/2P5/2N3P1/PP1PPP1P/R1BQKBNR b KQkq - 0 3,42,6,42,7,35,e2e4,"[{'rank': 1, 'move_uci': 'e2e4', 'eval_cp': 42, 'line_uci': ['e2e4', 'd7d5', 'e4e5', 'd5d4', 'e5f6', 'd4c3', 'f6g7', 'c3d2', 'c1d2', 'f8g7', 'd1c2', 'b7b6',...",7,0.521627,0.503091,0.018536,178.8,175.5,3.3,opening,"[equal, quiet]",False,False,122.299832,26,3.299832,0,93.0,False,False,False,False,False,False,False,A17,"English Opening: Anglo-Indian Defense, Hedgehog System"


In [18]:
pgn_text = game_data["pgn"]
game = chess.pgn.read_game(io.StringIO(pgn_text))

board = game.board()

board_states = [board.copy()]
moves = []

for move in game.mainline_moves():
    moves.append(move)
    board.push(move)
    board_states.append(board.copy())

print(f"Total moves / plies: {len(moves)}")

Total moves / plies: 60


In [19]:
def board_to_png_image(
    board: chess.Board,
    *,
    lastmove: chess.Move | None = None,
    size: int = 480,
):
    svg_data = chess.svg.board(
        board=board,
        lastmove=lastmove,
        size=size,
    )

    png_bytes = cairosvg.svg2png(bytestring=svg_data.encode("utf-8"))
    return Image.open(io.BytesIO(png_bytes))

In [20]:
MOVE_INFO_FIELD_ORDER = [
    "ply",
    "move_number",
    "player_color",
    "player_username",
    "san",
    "uci",
    "phase",
    "position_type_tags",
    "tactical_position",
    "quiet_middlegame",
    "complexity",
    "number_of_legal_moves",
    "eval_volatility_among_top_engine_lines",
    "forcing_line_depth",
    "low_gap_between_top_moves",
    "engine_top_move_is_forcing",
    "move_is_threat",
    "best_move_uci",
    "top_engine_moves",
    "best_move_cp_gain",
    "eval_before_cp",
    "eval_after_cp",
    "eval_best_move_cp",
    "eval_played_move_cp",
    "move_cp_loss",
    "win_prob_before",
    "win_prob_after",
    "win_prob_loss",
    "clock_before",
    "clock_after",
    "clock_spent",
    "is_capture",
    "is_check",
    "is_castling",
    "is_promotion",
    "opening_eco",
    "opening_name",
    "game_url",
    "game_uuid",
    "fen_before",
    "fen_after",
]


def ordered_move_columns(df: pd.DataFrame) -> list[str]:
    preferred = [column for column in MOVE_INFO_FIELD_ORDER if column in df.columns]
    extras = [column for column in df.columns if column not in preferred]
    return preferred + extras


def format_move_value(value):
    if isinstance(value, list):
        if not value:
            return ""
        if all(isinstance(item, str) for item in value):
            return ", ".join(value)
        return value

    if pd.isna(value):
        return None

    return value


def get_move_info(ply_index: int):
    """
    ply_index:
        0 = initial position
        1 = after first move
        2 = after second move
        ...
    """
    if ply_index == 0:
        return {
            "Position": "Initial position",
            "Move": None,
        }

    row = moves_df.iloc[ply_index - 1]

    return {
        column: format_move_value(row[column])
        for column in ordered_move_columns(moves_df)
    }

In [21]:
move_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(board_states) - 1,
    step=1,
    description="Ply",
    continuous_update=False,
)

prev_button = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")

output = widgets.Output()


def render_position(ply_index: int):
    with output:
        clear_output(wait=True)

        board = board_states[ply_index]
        lastmove = moves[ply_index - 1] if ply_index > 0 else None

        image = board_to_png_image(board, lastmove=lastmove)
        display(image)

        info = get_move_info(ply_index)

        info_df = pd.DataFrame(
            [{"field": key, "value": value} for key, value in info.items()]
        )

        display(info_df)


def on_slider_change(change):
    if change["name"] == "value":
        render_position(change["new"])


def go_previous(_):
    move_slider.value = max(move_slider.min, move_slider.value - 1)


def go_next(_):
    move_slider.value = min(move_slider.max, move_slider.value + 1)


move_slider.observe(on_slider_change)

prev_button.on_click(go_previous)
next_button.on_click(go_next)

display(widgets.HBox([prev_button, next_button]))
display(move_slider)
display(output)

render_position(0)

IntSlider(value=0, continuous_update=False, description='Ply', max=60)

Output()

In [22]:
moves_df[ordered_move_columns(moves_df)]

,ply,move_number,player_color,player_username,san,uci,phase,position_type_tags,tactical_position,quiet_middlegame,complexity,number_of_legal_moves,eval_volatility_among_top_engine_lines,forcing_line_depth,low_gap_between_top_moves,engine_top_move_is_forcing,move_is_threat,best_move_uci,top_engine_moves,best_move_cp_gain,eval_before_cp,eval_after_cp,eval_best_move_cp,eval_played_move_cp,move_cp_loss,win_prob_before,win_prob_after,win_prob_loss,clock_before,clock_after,clock_spent,is_capture,is_check,is_castling,is_promotion,opening_eco,opening_name,game_url,game_uuid,fen_before,fen_after,in_opening_book
0,1,1,white,bassisw,c4,c2c4,opening,"[equal, quiet]",False,False,121.092121,20,9.092121,0,92.0,False,False,e2e4,"[{'rank': 1, 'move_uci': 'e2e4', 'eval_cp': 45, 'line_uci': ['e2e4', 'c7c5', 'g1f3', 'e7e6', 'b1c3', 'b8c6', 'd2d4', 'c5d4', 'f3d4', 'g8f6', 'd4c6', 'b7c6']...",8,45,12,45,11,34,0.523170,0.506183,0.016987,180.0,180.0,0.0,False,False,False,False,A10,English Opening,https://www.chess.com/game/live/130193958939,40fc279d-cec9-11ef-ad47-6cfe544c0428,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1,rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR b KQkq - 0 1,True
1,2,1,black,Sylvathur,Nf6,g8f6,opening,"[equal, quiet]",False,False,110.801234,20,10.801234,0,80.0,False,False,e7e5,"[{'rank': 1, 'move_uci': 'e7e5', 'eval_cp': 15, 'line_uci': ['e7e5', 'b1c3', 'g8f6', 'g1f3', 'b8c6', 'g2g3', 'd7d5', 'c4d5', 'f6d5', 'd2d3']}, {'rank': 2, '...",20,15,38,15,41,26,0.491631,0.478809,0.012822,180.0,180.0,0.0,False,False,False,False,A15,English Opening: Anglo-Indian Defense,https://www.chess.com/game/live/130193958939,40fc279d-cec9-11ef-ad47-6cfe544c0428,rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR b KQkq - 0 1,rnbqkb1r/pppppppp/5n2/8/2P5/8/PP1PPPPP/RNBQKBNR w KQkq - 1 2,True
2,3,2,white,bassisw,Nc3,b1c3,opening,"[equal, quiet]",False,False,127.338708,22,10.338708,0,95.0,False,False,d2d4,"[{'rank': 1, 'move_uci': 'd2d4', 'eval_cp': 41, 'line_uci': ['d2d4', 'e7e6', 'b1c3', 'f8b4', 'e2e3', 'b4c3', 'b2c3', 'b7b6', 'f2f3', 'd7d6', 'e3e4', 'b8c6',...",5,41,20,41,20,21,0.521113,0.510304,0.010809,180.0,178.8,1.2,False,False,False,False,A16,"English Opening: Anglo-Indian Defense, Queen's Knight Variation",https://www.chess.com/game/live/130193958939,40fc279d-cec9-11ef-ad47-6cfe544c0428,rnbqkb1r/pppppppp/5n2/8/2P5/8/PP1PPPPP/RNBQKBNR w KQkq - 1 2,rnbqkb1r/pppppppp/5n2/8/2P5/2N5/PP1PPPPP/R1BQKBNR b KQkq - 2 2,True
3,4,2,black,Sylvathur,e6,e7e6,opening,"[equal, quiet]",False,False,114.847546,22,6.847546,0,86.0,False,False,e7e5,"[{'rank': 1, 'move_uci': 'e7e5', 'eval_cp': 20, 'line_uci': ['e7e5', 'g1f3', 'b8c6', 'g2g3', 'd7d5', 'c4d5', 'f6d5', 'f1g2', 'd5b6', 'd2d3', 'f8e7', 'c1e3',...",14,20,41,20,37,17,0.488842,0.477138,0.011704,180.0,178.8,1.2,False,False,False,False,A17,"English Opening: Anglo-Indian Defense, Hedgehog System",https://www.chess.com/game/live/130193958939,40fc279d-cec9-11ef-ad47-6cfe544c0428,rnbqkb1r/pppppppp/5n2/8/2P5/2N5/PP1PPPPP/R1BQKBNR b KQkq - 2 2,rnbqkb1r/pppp1ppp/4pn2/8/2P5/2N5/PP1PPPPP/R1BQKBNR w KQkq - 0 3,True
4,5,3,white,bassisw,g3,g2g3,opening,"[equal, quiet]",False,False,122.299832,26,3.299832,0,93.0,False,False,e2e4,"[{'rank': 1, 'move_uci': 'e2e4', 'eval_cp': 42, 'line_uci': ['e2e4', 'd7d5', 'e4e5', 'd5d4', 'e5f6', 'd4c3', 'f6g7', 'c3d2', 'c1d2', 'f8g7', 'd1c2', 'b7b6',...",7,42,6,42,7,35,0.521627,0.503091,0.018536,178.8,175.5,3.3,False,False,False,False,A17,"English Opening: Anglo-Indian Defense, Hedgehog System",https://www.chess.com/game/live/130193958939,40fc279d-cec9-11ef-ad47-6cfe544c0428,rnbqkb1r/pppp1ppp/4pn2/8/2P5/2N5/PP1PPPPP/R1BQKBNR w KQkq - 0 3,rnbqkb1r/pppp1ppp/4pn2/8/2P5/2N3P1/PP1PPP1P/R1BQKBNR b KQkq - 0 3,False
5,6,3,black,Sylvathur,c5,c7c5,opening,"[equal, quiet]",False,False,123.653837,28,8.653837,0,87.0,False,False,d7d5,"[{'rank': 1, 'move_uci': 'd7d5', 'eval_cp': 6, 'line_uci': ['d7d5', 'd2d4', 'd5c4', 'g1f3', 'c7c5', 'f1g2', 'b8c6', 'd1a4', 'c5d4', 'f3d4', 'd8d4', 'g2c6', ...",13,6,39,6